# Entrenamiento del modelo – Práctica MLOps

Este notebook forma parte del flujo **Colab → GitHub → procesar_datos.py → entrenamiento.ipynb → modelo.joblib → Gradio / FastAPI / Dashboard**.

Objetivo: predecir la columna `resultado` (**SI / NO**) a partir de variables agroclimáticas
(`temperatura`, `humedad`, `radiacion_solar`, `precipitacion`, `viento`) y el tipo de `cultivo`.

> **Nota:** como la variable objetivo es binaria (SI/NO) se usa **`LogisticRegression`** (modelo lineal de clasificación)
> en lugar de `LinearRegression`, que es para variables continuas. Así el modelo devuelve probabilidades y permite
> construir la **matriz de confusión** y las métricas que consume el dashboard.

Artefactos generados:

| Artefacto | Descripción |
|-----------|-------------|
| `models/modelo.joblib` | Pipeline completo (pre-procesamiento + modelo) serializado |
| `models/metricas.json` | Métricas, matriz de confusión, curva ROC, coeficientes y matriz de correlación |
| `output/matriz_confusion.png` | Gráfico de la matriz de confusión (hold-out) |
| `output/heatmap_correlacion.png` | Mapa de calor de correlaciones |


In [1]:
# ── 1. Librerías y rutas ─────────────────────────────────────────────────────
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")  # backend sin pantalla (Colab / CI)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# El notebook puede ejecutarse desde /notebooks (local) o desde la raíz del proyecto (Colab)
CWD = Path.cwd()
BASE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
DATA_PATH = BASE_DIR / "data" / "datos.csv"
MODELS_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "output"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_NUM = ["temperatura", "humedad", "radiacion_solar", "precipitacion", "viento"]
FEATURE_CAT = "cultivo"
TARGET = "resultado"
RANDOM_STATE = 42

print("Directorio base :", BASE_DIR)
print("Dataset         :", DATA_PATH, "| existe:", DATA_PATH.exists())
print("scikit-learn    :", sklearn.__version__)

Directorio base : C:\users\lreyn\downloads\practica_mlops_arena_ai
Dataset         : C:\users\lreyn\downloads\practica_mlops_arena_ai\data\datos.csv | existe: True
scikit-learn    : 1.9.1


In [2]:
# ── 2. Carga y exploración rápida ────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df[TARGET] = df[TARGET].astype(str).str.strip().str.upper()
df = df.dropna(subset=FEATURES_NUM + [FEATURE_CAT, TARGET]).reset_index(drop=True)

print("Forma:", df.shape)
display(df.head())
print("\nDistribución del objetivo:")
print(df[TARGET].value_counts())
print("\nTasa de SI por cultivo:")
print(df.groupby(FEATURE_CAT)[TARGET].apply(lambda s: (s == "SI").mean()).round(3))
display(df[FEATURES_NUM].describe().round(2))

Forma: (50, 8)


,id,cultivo,temperatura,humedad,radiacion_solar,precipitacion,viento,resultado
0,1,uva,19.6,66.0,18.7,1.1,2.4,NO
1,2,uva,30.5,43.0,21.3,0.2,4.1,NO
2,3,uva,26.4,59.6,25.7,5.6,6.9,SI
3,4,cafe,22.7,89.3,8.1,15.1,7.1,NO
4,5,cafe,20.2,71.5,18.7,4.5,4.1,SI



Distribución del objetivo:
resultado
NO    40
SI    10
Name: count, dtype: int64

Tasa de SI por cultivo:
cultivo
cafe    0.385
papa    0.053
uva     0.222
Name: resultado, dtype: float64


,temperatura,humedad,radiacion_solar,precipitacion,viento
count,50.00,50.00,50.00,50.00,50.00
mean,20.62,69.26,18.56,7.64,6.80
std,4.24,14.13,4.54,4.93,3.22
min,10.10,41.80,8.10,0.20,1.40
25%,18.60,61.65,15.60,4.20,4.28
50%,20.30,71.45,18.45,6.00,6.60
75%,22.62,78.45,21.25,11.78,8.62
max,30.90,94.90,30.00,17.60,14.30


In [3]:
# ── 3. Mapa de calor de correlaciones ────────────────────────────────────────
df_corr = df[FEATURES_NUM].copy()
df_corr["resultado_bin"] = (df[TARGET] == "SI").astype(int)
matriz_corr = df_corr.corr().round(3)

plt.figure(figsize=(7, 5.5))
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación (Pearson)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_correlacion.png", dpi=120)
plt.show()
matriz_corr

C:\Users\lreyn\AppData\Local\Temp\ipykernel_2884\49503938.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,temperatura,humedad,radiacion_solar,precipitacion,viento,resultado_bin
temperatura,1.000,-0.464,0.303,-0.248,-0.036,0.249
humedad,-0.464,1.000,-0.411,0.467,-0.139,0.005
radiacion_solar,0.303,-0.411,1.000,-0.373,-0.118,0.137
precipitacion,-0.248,0.467,-0.373,1.000,-0.045,0.007
viento,-0.036,-0.139,-0.118,-0.045,1.000,-0.113
resultado_bin,0.249,0.005,0.137,0.007,-0.113,1.000


In [4]:
# ── 4. División train / test y pipeline ──────────────────────────────────────
X = df[FEATURES_NUM + [FEATURE_CAT]]
y = (df[TARGET] == "SI").astype(int)          # 1 = SI, 0 = NO

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

preprocesador = ColumnTransformer([
    ("num", StandardScaler(), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), [FEATURE_CAT]),
])

modelo = Pipeline([
    ("pre", preprocesador),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
])

modelo.fit(X_train, y_train)
print(f"Entrenado con {len(X_train)} filas | test: {len(X_test)} filas | positivos en test: {int(y_test.sum())}")
modelo

Entrenado con 37 filas | test: 13 filas | positivos en test: 3


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['temperatura','humedad','radiacion_solar','precipitacion','viento', 'cultivo']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remai

In [5]:
# ── 5. Evaluación en hold-out: métricas + matriz de confusión ────────────────
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

cm_holdout = confusion_matrix(y_test, y_pred, labels=[0, 1])   # [[TN, FP], [FN, TP]]
metricas_holdout = {
    "accuracy":  round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
    "recall":    round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
    "f1":        round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
    "roc_auc":   round(float(roc_auc_score(y_test, y_proba)), 4),
    "matriz_confusion": cm_holdout.tolist(),
    "n": int(len(y_test)),
}
print(json.dumps(metricas_holdout, indent=2))
print(classification_report(y_test, y_pred, target_names=["NO", "SI"], zero_division=0))

fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm_holdout, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["NO", "SI"], yticklabels=["NO", "SI"], ax=ax)
ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); ax.set_title("Matriz de confusión (hold-out)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "matriz_confusion.png", dpi=120)
plt.show()

{
  "accuracy": 0.8462,
  "precision": 0.6,
  "recall": 1.0,
  "f1": 0.75,
  "roc_auc": 0.9,
  "matriz_confusion": [
    [
      8,
      2
    ],
    [
      0,
      3
    ]
  ],
  "n": 13
}
              precision    recall  f1-score   support

          NO       1.00      0.80      0.89        10
          SI       0.60      1.00      0.75         3

    accuracy                           0.85        13
   macro avg       0.80      0.90      0.82        13
weighted avg       0.91      0.85      0.86        13



C:\Users\lreyn\AppData\Local\Temp\ipykernel_2884\1478273287.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── 6. Validación cruzada estratificada (predicciones out-of-fold) ───────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_oof = cross_val_predict(modelo, X, y, cv=skf)
p_oof = cross_val_predict(modelo, X, y, cv=skf, method="predict_proba")[:, 1]

cm_cv = confusion_matrix(y, y_oof, labels=[0, 1])
metricas_cv = {
    "n_splits": 5,
    "accuracy":  round(float(accuracy_score(y, y_oof)), 4),
    "precision": round(float(precision_score(y, y_oof, zero_division=0)), 4),
    "recall":    round(float(recall_score(y, y_oof, zero_division=0)), 4),
    "f1":        round(float(f1_score(y, y_oof, zero_division=0)), 4),
    "roc_auc":   round(float(roc_auc_score(y, p_oof)), 4),
    "matriz_confusion": cm_cv.tolist(),
    "n": int(len(y)),
}
print(json.dumps(metricas_cv, indent=2))

fpr, tpr, _ = roc_curve(y, p_oof)
plt.figure(figsize=(4.5, 4))
plt.plot(fpr, tpr, label=f"ROC (AUC = {metricas_cv['roc_auc']:.2f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Curva ROC (out-of-fold)"); plt.legend()
plt.tight_layout(); plt.show()

{
  "n_splits": 5,
  "accuracy": 0.62,
  "precision": 0.2353,
  "recall": 0.4,
  "f1": 0.2963,
  "roc_auc": 0.61,
  "matriz_confusion": [
    [
      27,
      13
    ],
    [
      6,
      4
    ]
  ],
  "n": 50
}


C:\Users\lreyn\AppData\Local\Temp\ipykernel_2884\3342273888.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [7]:
# ── 7. Importancia de variables (coeficientes del modelo lineal) ─────────────
nombres = modelo.named_steps["pre"].get_feature_names_out()
coefs = modelo.named_steps["clf"].coef_[0]
coeficientes = {str(n).replace("num__", "").replace("cat__", ""): round(float(c), 4)
                for n, c in zip(nombres, coefs)}
pd.Series(coeficientes).sort_values().plot(kind="barh", figsize=(6, 3.5), title="Coeficientes (log-odds de SI)")
plt.tight_layout(); plt.show()
coeficientes

C:\Users\lreyn\AppData\Local\Temp\ipykernel_2884\661821699.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


{'temperatura': 0.6224,
 'humedad': 0.3147,
 'radiacion_solar': 0.4333,
 'precipitacion': -0.237,
 'viento': 0.1513,
 'cultivo_cafe': 0.6507,
 'cultivo_papa': -0.3072,
 'cultivo_uva': -0.3434}

In [8]:
# ── 8. Serialización: modelo.joblib + metricas.json ──────────────────────────
ruta_modelo = MODELS_DIR / "modelo.joblib"
joblib.dump(modelo, ruta_modelo)

metricas = {
    "modelo": "LogisticRegression",
    "version": "1.0.0",
    "fecha_entrenamiento": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "features_numericas": FEATURES_NUM,
    "feature_categorica": FEATURE_CAT,
    "categorias_cultivo": sorted(df[FEATURE_CAT].unique().tolist()),
    "target": TARGET,
    "clases": ["NO", "SI"],
    "n_filas": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "holdout": metricas_holdout,
    "cv": metricas_cv,
    "roc_curve": {"fpr": [round(float(v), 4) for v in fpr], "tpr": [round(float(v), 4) for v in tpr]},
    "coeficientes": coeficientes,
    "intercepto": round(float(modelo.named_steps["clf"].intercept_[0]), 4),
    "medias_entrenamiento": {c: round(float(X_train[c].mean()), 4) for c in FEATURES_NUM},
    "desviaciones_entrenamiento": {c: round(float(X_train[c].std()), 4) for c in FEATURES_NUM},
    "rangos": {c: [round(float(df[c].min()), 2), round(float(df[c].max()), 2)] for c in FEATURES_NUM},
    "matriz_correlacion": {"columnas": matriz_corr.columns.tolist(), "valores": matriz_corr.values.tolist()},
}
with open(MODELS_DIR / "metricas.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

print("Modelo guardado en   :", ruta_modelo, f"({ruta_modelo.stat().st_size/1024:.1f} KB)")
print("Métricas guardadas en:", MODELS_DIR / "metricas.json")

Modelo guardado en   : C:\users\lreyn\downloads\practica_mlops_arena_ai\models\modelo.joblib (3.3 KB)
Métricas guardadas en: C:\users\lreyn\downloads\practica_mlops_arena_ai\models\metricas.json


In [9]:
# ── 9. Prueba de carga del artefacto (lo mismo que hacen Gradio y FastAPI) ───
modelo_cargado = joblib.load(ruta_modelo)
ejemplo = pd.DataFrame([{
    "temperatura": 21.0, "humedad": 80.0, "radiacion_solar": 16.0,
    "precipitacion": 12.0, "viento": 5.0, "cultivo": "cafe",
}])
prob_si = float(modelo_cargado.predict_proba(ejemplo)[0, 1])
print(f"Predicción: {'SI' if prob_si >= 0.5 else 'NO'}  (P(SI) = {prob_si:.3f})")

Predicción: SI  (P(SI) = 0.560)
